# Ghana Voice: train a voice for a Ghanaian language

Runs on a free Colab GPU (**Runtime → Change runtime type → T4 GPU**). Each step is one cell; run them in order.

| Language | Code | Starts from | Open data |
|---|---|---|---|
| Twi (Asante) | `tw` | MMS Akan | 200 h |
| Twi (Akuapem) | `ak` | MMS Akan | 63 h |
| Fante | `fat` | MMS Akan | yes |
| Ewe | `ee` | MMS Ewe | yes |
| Hausa | `ha` | MMS Hausa | yes |
| Dagbani | `dag` | MMS Mampruli (closest relative) | yes |
| Ga | `ga` | MMS Ewe | **no: record your own** (see speech/README.md) |

`python serve.py list` shows all 21 languages. Training data and MMS voices are CC BY-NC 4.0 (non-commercial use).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
If the repository is private, put a GitHub token in `TOKEN` (Settings → Developer settings → Fine-grained token, read-only).

In [ ]:
TOKEN = ""   # only for a private repository
BRANCH = "main"
auth = f"{TOKEN}@" if TOKEN else ""
!git clone --depth 1 -b {BRANCH} https://{auth}github.com/NuelNexus/ghana-poultry-vision.git
%cd ghana-poultry-vision/speech
!bash setup_training.sh

## 2. Pick the language and collect recordings
About 6 hours of speech is a good start; it downloads only what it needs (roughly 1 GB).

In [ ]:
LANG = "ee"
HOURS = 6
!python prepare_data.py --lang {LANG} --hours {HOURS}

## 3. Build the starting voice
Converts the MMS voice into a trainable checkpoint and adds any letters it never saw.

In [ ]:
!python make_base.py --lang {LANG}

## 4. Train
8,000 steps take roughly 3–4 hours on a T4. For a quick first listen use `--steps 1500`. If the GPU runs out of memory, add `--batch 8`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
!python train.py --lang {LANG} --steps 8000

## 5. Listen

In [ ]:
import sys; sys.path.insert(0, ".")
from ghana_voice import languages
sample = languages.get(LANG).sample
!GHANA_VOICE_MODELS=models GHANA_VOICE_LANGS={LANG} python serve.py say "{sample}" --lang {LANG} --out sample.wav
from IPython.display import Audio
Audio("sample.wav")

## 6. Keep the voice
Either download the folder, or upload it to your Hugging Face account.

In [ ]:
!cd models && zip -qr ../ghana-voice-{LANG}.zip {LANG}
from google.colab import files
files.download(f"ghana-voice-{LANG}.zip")

In [ ]:
# Optional: upload to Hugging Face (needs a write token from huggingface.co/settings/tokens)
from huggingface_hub import login, upload_folder
login()
upload_folder(folder_path=f"models/{LANG}", repo_id=f"YOUR-NAME/ghana-voice-{LANG}", repo_type="model")

## Use it in PoultryGrid
Unzip into `speech/models/<code>/` on the Pi or PC that runs the voice server, then:
```
cd speech
GHANA_VOICE_MODELS=models GHANA_VOICE_TOKEN=pick-a-secret python serve.py serve --host 0.0.0.0 --port 5003
```
and set `GHANA_VOICE_URL=http://<that-machine>:5003` and the same `GHANA_VOICE_TOKEN` in the app's `.env`.